In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_canal_venta")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_canal_venta (
  id_canal_venta LONG,
  nombre_canal_venta STRING
)
""")


In [0]:
import pandas as pd

df = spark.table("workspace.silver.adidas_us_sales").toPandas()

# Se crea la dimension de tiendas, desde la tabla de ventas
dim_canal_venta = df[['canal_venta']].drop_duplicates().reset_index(drop=True)
dim_canal_venta['id_canal_venta'] = dim_canal_venta.index + 1
dim_canal_venta = dim_canal_venta.rename(columns={'canal_venta': 'nombre_canal_venta'})
dim_canal_venta = dim_canal_venta[['id_canal_venta', 'nombre_canal_venta']]

df_spark = spark.createDataFrame(dim_canal_venta)

In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_canal_venta")